# Working with Redis for Chat Message History

This notebook demonstrates how to use the `RedisChatMessageHistory` class from the langchain-redis package to store and manage chat message history using Redis.

## Installation



In [ ]:
# %pip install ipywidgets
# %pip install langchain-redis
# %pip install langchain-openai
# %pip install redis

## Importing Required Libraries

In [1]:
import os
import redis

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_openai import ChatOpenAI
from langchain_redis import RedisChatMessageHistory

## Setting up Redis Connection

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

redis_url = os.getenv("REDIS_URL")
redis_client = redis.from_url(redis_url)
redis_client.ping()

True

## Set the OpenAI API key

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")

## Working with Chat Message History Directly

In [5]:
history = RedisChatMessageHistory(
    session_id="bob_123", redis_url=redis_url
)

history.add_user_message("Hello, AI assistant!")
history.add_ai_message("Hello! How can I assist you today?")

print("Chat History:")
for message in history.messages:
    print(f"{type(message).__name__}: {message.content}")

Chat History:
HumanMessage: Hello, AI assistant!
AIMessage: Hello! How can I assist you today?


## Chat Message History with Language Models

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

llm = ChatOpenAI()
chain = prompt | llm

def get_redis_history(session_id: str) -> BaseChatMessageHistory:
    return RedisChatMessageHistory(session_id, redis_url=redis_url)

chain_with_history = RunnableWithMessageHistory(
    chain, get_redis_history, input_messages_key="input", history_messages_key="history"
)

response1 = chain_with_history.invoke(
    {"input": "Hi, my name is Alice."},
    config={"configurable": {"session_id": "alice_123"}},
)
print("AI Response 1:", response1.content)

response2 = chain_with_history.invoke(
    {"input": "What's my name?"}, config={"configurable": {"session_id": "alice_123"}}
)
print("AI Response 2:", response2.content)

response3 = chain_with_history.invoke(
    {"input": "What's my name?"}, config={"configurable": {"session_id": "jane_123"}}
)
print("AI Response 3:", response3.content)

c:\Users\samin\Python\REDIS\langchain-apps-with-redis-main\langchain-apps-with-redis-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AI Response 1: Hello, Alice! It's nice to meet you. How can I assist you today?
AI Response 2: Your name is Alice.
AI Response 3: I'm sorry, I don't have access to personal information about users. How can I assist you today?


## Cleanup

In [7]:
# Clear the chat history
history.clear()

# Clear the chat history for specific sessions
session_id = "alice_123"
history = RedisChatMessageHistory(session_id=session_id, redis_url=redis_url)
history.clear()

session_id = "jane_123"
history = RedisChatMessageHistory(session_id=session_id, redis_url=redis_url)
history.clear()

print("Chat history cleared")

Chat history cleared
